In [13]:
from pathlib import Path
from typing import TypedDict
import subprocess
import uuid

from dotenv import load_dotenv

from supabase import create_client, Client

from langchain_openai import ChatOpenAI

from langgraph.graph import (
    StateGraph,
    START,
    END,
)

from IPython.display import Audio, display
import os

In [2]:
load_dotenv()

True

In [5]:


SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase: Client = create_client(
    SUPABASE_URL,
    SUPABASE_KEY,
)

print("Supabase connected")

Supabase connected


In [ ]:
thread = supabase.table("threads").insert({
    "title": "My First Chat"
}).execute()

print(thread.data)
thread_id = thread.data[0]["id"]

print("Thread ID:", thread_id)

[{'id': 'f3565232-0628-4bb2-b2ac-915fd2936f9a', 'user_id': None, 'title': 'My First Chat', 'created_at': '2026-08-19T19:55:05.973422+00:00', 'updated_at': '2026-08-19T19:55:05.973422+00:00'}]


In [8]:
message = supabase.table("messages").insert({
    "thread_id": thread_id,
    "role": "user",
    "content": "Hello, how are you?"
}).execute()

print(message.data)

[{'id': '786990c6-4240-4191-a119-36ae69cef35d', 'thread_id': 'f3565232-0628-4bb2-b2ac-915fd2936f9a', 'role': 'user', 'content': 'Hello, how are you?', 'audio_url': None, 'created_at': '2026-08-19T19:55:34.390763+00:00'}]


In [9]:
messages = (
    supabase
    .table("messages")
    .select("*")
    .eq("thread_id", thread_id)
    .order("created_at")
    .execute()
)

print(messages.data)

[{'id': '786990c6-4240-4191-a119-36ae69cef35d', 'thread_id': 'f3565232-0628-4bb2-b2ac-915fd2936f9a', 'role': 'user', 'content': 'Hello, how are you?', 'audio_url': None, 'created_at': '2026-08-19T19:55:34.390763+00:00'}]


In [10]:
class ChatState(TypedDict):
    thread_id: str
    user_message: str
    llm_response: str
    audio_file: str

In [14]:
llm = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct",
    # model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [15]:
def chat_node(state: ChatState):

    thread_id = state["thread_id"]
    user_message = state["user_message"]

    # Get previous conversation
    previous_messages = (
        supabase
        .table("messages")
        .select("role, content")
        .eq("thread_id", thread_id)
        .order("created_at")
        .execute()
    )

    messages = []

    for message in previous_messages.data:

        messages.append(
            (
                message["role"],
                message["content"]
            )
        )

    # Add current user message
    messages.append(
        ("user", user_message)
    )

    # Send conversation to LLM
    response = llm.invoke(messages)

    response_text = response.content

    # Save user message
    supabase.table("messages").insert({
        "thread_id": thread_id,
        "role": "user",
        "content": user_message
    }).execute()

    # Save assistant message
    supabase.table("messages").insert({
        "thread_id": thread_id,
        "role": "assistant",
        "content": response_text
    }).execute()

    return {
        "llm_response": response_text
    }

In [17]:
PROJECT_ROOT = Path.cwd().parent

AUDIO_DIR = PROJECT_ROOT / "backend" / "audio"

PIPER_EXECUTABLE = (
    PROJECT_ROOT
    / ".venv"
    / "Scripts"
    / "piper.exe"
)

PIPER_MODEL = (
    PROJECT_ROOT
    / "backend"
    / "models"
    / "en_US-lessac-medium.onnx"
)

AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Piper:", PIPER_EXECUTABLE.exists())
print("Model:", PIPER_MODEL.exists())

Piper: True
Model: True


In [18]:
def piper_node(state: ChatState):

    text = state["llm_response"]

    filename = f"{uuid.uuid4()}.wav"

    output_path = AUDIO_DIR / filename

    command = [
        str(PIPER_EXECUTABLE),
        "--model",
        str(PIPER_MODEL),
        "--output_file",
        str(output_path),
    ]

    subprocess.run(
        command,
        input=text,
        text=True,
        check=True,
    )

    return {
        "audio_file": str(output_path)
    }

In [19]:
builder = StateGraph(ChatState)

builder.add_node(
    "chat",
    chat_node
)

builder.add_node(
    "piper",
    piper_node
)

builder.add_edge(
    START,
    "chat"
)

builder.add_edge(
    "chat",
    "piper"
)

builder.add_edge(
    "piper",
    END
)

graph = builder.compile()

In [20]:
def create_thread(title="New Chat"):

    result = (
        supabase
        .table("threads")
        .insert({
            "title": title
        })
        .execute()
    )

    return result.data[0]["id"]

In [21]:
user_input = "Tell me about yourself"

result = graph.invoke({
    "thread_id": thread_id,
    "user_message": user_input,
})

In [22]:
display(
    Audio(result["audio_file"])
)

In [23]:
print("User:")
print(result["user_message"])

print("\nAssistant:")
print(result["llm_response"])

print("\nAudio:")
print(result["audio_file"])

User:
Tell me about yourself

Assistant:
I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."

Audio:
c:\Users\MuneebZafar\Downloads\Personal\Langchain\Projects\Voice Chatbot\backend\audio\e353c537-9266-461d-8194-42a485838d28.wav
